<a href="https://colab.research.google.com/github/Eswa2020/urban-parking-prediction-models/blob/master/I_Moran_Getis_Ord.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

###Random Forest Regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# -----------------------------
# 1️⃣ Load dataset
# -----------------------------
df = pd.read_csv("/content/dubai_model_with_poi2.csv")
# Standardize column names
df.columns = df.columns.str.strip().str.lower()
df.head()
# -----------------------------


In [ ]:
# 2️⃣ Handle missing values
# -----------------------------
df = df.replace([np.inf, -np.inf], np.nan)
df = df.fillna(0)


In [ ]:
# -----------------------------
# 3️⃣ Define target
# -----------------------------
y = df["capacity_density"]

# -----------------------------

In [ ]:
# 4️⃣ Define predictors
# -----------------------------
X = df.drop(columns=[
    "fid",
    "capacity_density",   # target
    "capacity_num",
    "capacity_num_sum"
])
# -----------------------------

In [ ]:
# 5️⃣ Remove leakage variables
# -----------------------------
leakage_cols = [c for c in X.columns if "capacity" in c.lower()]
print("Dropping leakage columns:", leakage_cols)

X = X.drop(columns=leakage_cols)

# -----------------------------

In [ ]:
# 6️⃣ Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------

In [ ]:
X = df[[
    "res_ratio",        # residential intensity
    "road_density",     # accessibility structure
    "hubdist",          # proximity to mobility hub
    "poi_density"       # activity intensity
]]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

corr = X.corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.title("Feature Correlation Matrix")
plt.show()

In [ ]:
#we do NOT have multicollinearity problems.
#That is good.

In [ ]:
# 7️⃣ Scale features (correct way)
# -----------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [ ]:
# -----------------------------
# 8️⃣ Random Forest Model
# -----------------------------
rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=10,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)
rf.fit(X_train_scaled, y_train)

# -----------------------------

In [ ]:
# -----------------------------
# 9️⃣ Predictions
# -----------------------------
y_pred_rf = rf.predict(X_test_scaled)

# -----------------------------

In [ ]:
# 🔟 Evaluation
# -----------------------------
mae  = mean_absolute_error(y_test, y_pred_rf)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2   = r2_score(y_test, y_pred_rf)

print("\nRANDOM FOREST RESULTS")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

# Adjusted R²
n = X_test.shape[0]
p = X_test.shape[1]
adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
print("Adjusted R²:", adj_r2)

# -----------------------------

In [ ]:
# 11️⃣ Cross-Validation
# -----------------------------
cv_scores = cross_val_score(rf, X_train_scaled, y_train, cv=5, scoring="r2")
print("\nCross-Validation R² mean:", cv_scores.mean())

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=10,
    random_state=42
)

importances = pd.Series(
    perm.importances_mean,
    index=X_test.columns
).sort_values(ascending=False)

print(importances)

In [ ]:
print(X_test.shape)
print(len(X.columns))
print(len(perm.importances_mean))

In [ ]:
# -----------------------------
# 14️⃣ Residual Diagnostics
# -----------------------------
residuals = y_test - y_pred_rf

plt.figure(figsize=(6,4))
plt.scatter(y_pred_rf, residuals)
plt.axhline(0)
plt.title("Residual Plot")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.tight_layout()
plt.show()

In [ ]:
# Define predictors again
X = active[["poi_density", "res_ratio", "road_density", "hubdist"]]
y = active["giz_binary"]

In [ ]:
X = active[["poi_density"]]

##XGBOOST Model

In [ ]:
# Remove constant columns
constant_cols = [c for c in X.columns if X[c].nunique() <= 1]
X = X.drop(columns=constant_cols)

# -----------------------------

In [ ]:
# 4️⃣ Train/Test Split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# -----------------------------

In [ ]:
from xgboost import XGBRegressor

# 5️⃣ XGBoost Model
# -----------------------------
xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42
)

xgb.fit(X_train, y_train)

# -----------------------------

In [ ]:

# 6️⃣ Predictions
# -----------------------------
y_pred_xgb = xgb.predict(X_test)

# -----------------------------

In [ ]:
# 7️⃣ Evaluation
# -----------------------------
mae  = mean_absolute_error(y_test, y_pred_xgb)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2   = r2_score(y_test, y_pred_xgb)

print("XGBOOST RESULTS")
print("MAE :", mae)
print("RMSE:", rmse)
print("R²  :", r2)

In [ ]:
# -----------------------------
# 12️⃣ Feature Importance
# -----------------------------
importances = pd.Series(xgb.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

print("\nTop Drivers:")
print(importances.head(10))


##Install dependencies

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np

In [ ]:
!pip install geopandas shapely pyproj rtree

In [ ]:
!pip install geopandas

In [ ]:

pois = gpd.read_file("/content/POIs_layer.gpkg")
parking = gpd.read_file("/content/Parking_Layer.gpkg")

In [ ]:
print(grid.crs)
print(pois.crs)
print(parking.crs)

In [ ]:
parking = parking.to_crs(grid.crs)

##Spatial Join (POI count/Capacity_sum)

In [ ]:
poi_join = gpd.sjoin(grid, pois, predicate="intersects", how="left")

print(poi_join.columns)

In [ ]:
# Count only real POI matches
poi_count = (
    poi_join[~poi_join["index_right"].isna()]
    .groupby(poi_join[~poi_join["index_right"].isna()].index)
    .size()
)

grid["poi_count"] = poi_count
grid["poi_count"] = grid["poi_count"].fillna(0)

In [ ]:
grid["poi_count"].describe()

Interpretation:

• 75% of grid cells have 0 POIs
• Mean ≈ 0.09
• Max = 69

That is realistic spatial sparsity.

In [ ]:
parking["CAPACITY_REAL"].sum()

In [ ]:
print("Grid CRS:", grid.crs)
print("Parking CRS:", parking.crs)

print("Grid bounds:", grid.total_bounds)
print("Parking bounds:", parking.total_bounds)

In [ ]:
# Step 1 — set correct original CRS
parking = parking.set_crs(epsg=4326, allow_override=True)

# Step 2 — reproject properly
parking = parking.to_crs(grid.crs)

# Confirm fix
print(parking.total_bounds)

In [ ]:
parking_join = gpd.sjoin(grid, parking, predicate="intersects", how="left")

capacity_sum = (
    parking_join[~parking_join["index_right"].isna()]
    .groupby(parking_join[~parking_join["index_right"].isna()].index)["CAPACITY_REAL"]
    .sum()
)

grid["capacity"] = capacity_sum
#grid["capacity"] = grid["capacity"].fillna(0)

In [ ]:
grid["capacity"].describe()

In [ ]:
grid[["poi_count", "capacity"]].describe()

Capacity summary:
* count = 1,148
* mean ≈ 356
* 25% = 90
* 50% = 277
* 75% = 546
* max = 1,928

This tells us:Only 1,148 grid cells intersect parking supply polygons.
The rest are NaN (not 0 — NaN).That is correct spatial logic.

In [ ]:
#Before scaling, we must clean the variables.
#poi_count → already 0 for empty cells (correct)
#capacity → NaN for cells with no parking
#decide modeling logic:
#If a grid cell has:A)no parking supply → that is real zero supply
#So we convert NaN → 0

In [ ]:
grid["capacity"] = grid["capacity"].fillna(0)

In [ ]:
grid[["poi_count", "capacity"]].describe()

##Normalize

In [ ]:
from sklearn.preprocessing import StandardScaler

features = grid[["poi_count", "capacity"]]

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

grid["poi_z"] = scaled[:, 0]
grid["capacity_z"] = scaled[:, 1]

In [ ]:
grid[["poi_z", "capacity_z"]].describe()

In [ ]:
#Mean should be ~0
#Std should be ~1....means is ok

#Now we compute a clean supply–demand delta:

In [ ]:
grid["delta_z"] = grid["poi_z"] - grid["capacity_z"]

In [ ]:
grid["delta_z"].describe()

In [ ]:
#above output is statistically valid — but it reveals something important about your spatial structure.Meaning......
#Count = 152,322
#Mean ≈ 0 → correct after standardization
#Std = 1.318 → expected because delta is difference of two standardized variables
#Now the critical part:
#25%, 50%, 75% are all the same value:-0.05248
#That tells us:At least 75% of grid cells have identical structure:a(poi_count = 0 and b)capacity = 0
#After scaling, those identical zeros became identical z-scores.Their subtraction produces the same delta.means the grid is extremely sparse.It true(DXB)spatial reality.

In [ ]:
#aLSO extreme values:A)min = -44.99 & B)max = 87.20
#These are strong outliers.
#meaning...?
#Some grid cells have extremely high POI relative to supply
#Some have extremely high supply relative to demand

##Rule-Based Zoning (Baseline Labels)

In [ ]:
#Before modeling, we must stabilize this distribution.

In [ ]:
#inspect distribution first visually
import matplotlib.pyplot as plt

plt.hist(grid["delta_z"], bins=100)
plt.show()

In [ ]:
#huge spike around ~ -0.05 & long heavy tails
#This is typical in urban spatial analysis.
#to prevent a few extreme cells from dominating clustering.
# we Clip extreme outliers to avoid ML instability.

In [ ]:
grid["delta_clipped"] = grid["delta_z"].clip(-5, 5)

In [ ]:
grid["delta_clipped"].describe()

In [ ]:
#Notice:a) 75% of cells still cluster around −0.052,b)Extreme tails are clipped & c)Std ≈ 0.586 → reasonable spread
#to define rule-based zoning We will use symmetric thresholds around zero.why?.....Because:
#delta_z = demand_z − capacity_z
#Zero = perfect balance

#Implementing Zoning

In [ ]:
#to define Threshold Logic we choose:
#a)Deficit Zone → delta_clipped > +0.5
#b)Oversupply Zone → delta_clipped < −0.5
#c)Balanced Zone → between −0.5 and +0.5
#......Why ±0.5?....Because:
#a)In z-space, 0.5 ≈ moderate deviation from mean.
#b)It avoids labeling everything as balanced.
#simply meaning:“Zones exceeding half a standard deviation were classified as imbalanced.”

In [ ]:
def rule_zone(delta):
    if delta > 0.5:
        return "Deficit"
    elif delta < -0.5:
        return "Oversupply"
    else:
        return "Balanced"

grid["zone_rule"] = grid["delta_clipped"].apply(rule_zone)

In [ ]:
#check distribution
grid["zone_rule"].value_counts(normalize=True)

In [ ]:
#check for extremes also
grid.groupby("zone_rule")[["poi_count","capacity"]].mean()

In [ ]:
#Deficit:
#POI ≈ 2.2
#Capacity ≈ 1.39
#Demand > Supply → correct.
#Oversupply:
#POI ≈ 1.41
#Capacity ≈ 480

That basically means:
Almost no demand and almost no supply.
So “Balanced” here mostly means “inactive desert / empty grid”.
That is structurally true but analytically misleading.



Massive supply concentration → correct.

So the rule logic is functioning mathematically.

But the Balanced class is being dominated by empty cells.

This is the spatial sparsity problem again.

In [ ]:
#we dont model dominated by empty land.We excluded Grid cells with neither demand nor supply to prevent imbalance classification which can lead to artificial balance inflation.

In [ ]:
#creating subset of only active grid
active = grid[(grid["poi_count"] > 0) | (grid["capacity"] > 0)].copy()

In [ ]:
#check distribution
active["zone_rule"].value_counts(normalize=True)

In [ ]:
active.groupby("zone_rule")[["poi_count","capacity"]].mean()

In [ ]:
#Interpretation:
#Balanced A)POI ≈ 2.97 & B)Capacity ≈ 158
#Deficit A)POI ≈ 2.20 & B)Capacity ≈ 1.39
#Oversupply A)POI ≈ 1.41 &B)Capacity ≈ 480
#Now structure makes sense:
#Deficit → demand present, almost no supply
#Oversupply → heavy supply concentration
#Balanced → moderate demand, moderate supply(Balanced is no longer just desert.)

##KMeans Clustering (Unsupervised)

In [ ]:
#We cluster only the active subset.
active = grid[(grid["poi_count"] > 0) | (grid["capacity"] > 0)].copy()

In [ ]:
#must standardize again
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
active_scaled = scaler.fit_transform(active[["poi_count", "capacity"]])

active["poi_z"] = active_scaled[:, 0]
active["capacity_z"] = active_scaled[:, 1]

In [ ]:
#start with 3 clusters to compare with zone based ruling
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=3, random_state=42)
active["cluster"] = kmeans.fit_predict(active[["poi_z", "capacity_z"]])

In [ ]:
#Evaluate cluster
from sklearn.metrics import silhouette_score

score = silhouette_score(active[["poi_z", "capacity_z"]], active["cluster"])
print("Silhouette score:", score)

In [ ]:
#Silhouette score = 0.808That is extremely high.
#means the feature space (poi_count, capacity) forms very well-separated groups.
#clustering structure is not random.

In [ ]:
#explaining clusters
active.groupby("cluster")[["poi_count","capacity"]].mean()

In [ ]:
#Cluster 0 ---A)POI ≈ 1.88 & B)Capacity ≈ 13
#meaning.....Moderate demand, low–moderate supply.
#Cluster 1--- A)POI ≈ 1.81 & B)Capacity ≈ 677
#meaning.....Supply-dominant cluster.
#Cluster 2---A)POI ≈ 27.98 & B)Capacity ≈ 2.12
#meaning....Extreme demand concentration, minimal supply.

In [ ]:
#compare with rule-based zoning
pd.crosstab(active["zone_rule"], active["cluster"])

In [ ]:
#Zone Balanced:---Mostly cluster 0 & (29 vs almost none elsewhere)
#Zone Deficit:---Mostly cluster 0 and cluster 2....Cluster 2 aligns strongly with deficit (57 cells)
#Oversupply:---Mostly cluster 1 & (483 cells)
#This shows:
#Cluster 1 ≈ Oversupply
#Cluster 2 ≈ High-demand deficit core
#Cluster 0 ≈ Mixed / transitional

So ML is not contradicting rule-based zoning.
It is refining it.

###SPATIAL MAPPING Visualization)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(16, 6))

active.plot(column="zone_rule", categorical=True, legend=True, ax=ax[0])
ax[0].set_title("Rule-Based Zoning")

active.plot(column="cluster", categorical=True, legend=True, ax=ax[1])
ax[1].set_title("KMeans Clusters")

plt.show()

### GLOBAL MORAN’S I (Spatial Autocorrelation)

In [ ]:
!pip install libpysal esda

In [ ]:
#interpretation:
#If Moran’s I > 0 and p < 0.05 → delta imbalance is spatially clustered.(That means deficit and oversupply are not random.)

In [ ]:
import libpysal
from esda.moran import Moran

# Create spatial weights (Queen contiguity)
w = libpysal.weights.Queen.from_dataframe(active)
w.transform = "r"

# Moran’s I on delta
moran = Moran(active["delta_clipped"], w)

print("Moran's I:", moran.I)
print("p-value:", moran.p_sim)

In [ ]:
#the results:
#Moran’s I = 0.7037
#p-value = 0.001----->This is extremely strong spatial autocorrelation.
#Interpretation:A)Positive and high (close to 1) & B)Highly statistically significant
#This means:Demand–supply imbalance is NOT random.It is spatially clustered.
#That validates your entire modeling framework.

### GETIS-ORD Gi* (Hotspot Analysis)

In [ ]:
#This identifies where hotspots occur.
#Interpretation:
#GiZ > 1.96 → significant hotspot (deficit hotspot)
#GiZ < -1.96 → significant coldspot (oversupply cluster)

In [ ]:
from esda.getisord import G_Local

g_local = G_Local(active["delta_clipped"], w)

active["GiZ"] = g_local.Zs

active.plot(column="GiZ", cmap="coolwarm", legend=True)
plt.title("Getis-Ord Gi* Z-Scores")
plt.show()

In [ ]:
#🔴 Red clusters (positive GiZ)
#→ Statistically significant deficit hotspots
#→ Areas where demand pressure clusters spatially
#🔵 Blue cluster (negative GiZ)
#→ Statistically significant oversupply coldspot
#→ Concentrated parking supply cluster

#Spatially:(aligning with map axes are in UTM Zone 40N (EPSG:32640).)
#The northern coastal band shows a strong deficit concentration (red)--Y (Northing) ranges ≈ 2,740,000 – 2,800,000 meters
#A tight blue cluster indicates heavy supply concentration in a specific zone
#Interior areas are mostly neutral (not statistically significant)
#This means imbalance is not random — it forms coherent urban structures.
#This confirms Moran’s I = 0.70.

Now we have:
• Moran → Is clustering present?
• Getis → Where are statistically significant clusters?
---->the difference.A)Moran = global pattern & B)
Getis = local hotspot detection

###Random Forest Classifier

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

# Encode labels
active["zone_encoded"] = active["zone_rule"].astype("category").cat.codes
X = active_reg[[
    "road_density2",
    "res_ratio",
    "poi_z"
    "poi_density",
    "hubdist"
]]
X = active[[]]
y = active["zone_encoded"]

In [ ]:
#Split Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [ ]:
# -----------------------------
# Random Forest Classifier
# -----------------------------
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)


In [ ]:
#Run model
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)

In [ ]:
# Evaluation
# -----------------------------
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
#Overall Accuracy: 0.9985---That is effectively perfect classification.
#Class 1 and Class 2:a)Precision = 1.00 b)Recall = 1.00 and c)F1 = 1.00
#These correspond to the dominant zoning classes (Deficit and Oversupply).
#The model is separating them cleanly in feature space.
#Class 0:a)Support = 6,b)Precision = 0.71 and c)Recall = 0.83
#This is the Balanced class — very small sample size.(Only 6 test samples.)
#That is why its metrics fluctuate.Statistically irrelevant due to tiny support.
#Macro average ≈ 0.92 and Weighted average ≈ 1.00

###confirms:
#The rule-based zoning is highly separable.
#The ML model validates your zoning logic.

#This is not overfitting.It is deterministic structure.

#Now we synthesize the entire modeling stack....for RAG

### Random Classifier on Giz

create statistically meaningful classes:

In [ ]:
import numpy as np

# 0 = Not Significant
# 1 = Hotspot (Deficit cluster)
# 2 = Coldspot (Oversupply cluster)

active["giz_class"] = 0

active.loc[active["GiZ"] >= 1.96, "giz_class"] = 1
active.loc[active["GiZ"] <= -1.96, "giz_class"] = 2

In [ ]:
#standard practice check class balance
print(active["giz_class"].value_counts())
print(active["giz_class"].value_counts(normalize=True))

In [ ]:
#No Hotspots (GiZ ≥ 1.96)
#Only:Coldspots (oversupply clusters).....not significant
#No deficit hotspots were detected at 95% confidence.
#means:Parking imbalance in Dubai is spatially clustered toward oversupply concentration rather than deficit concentration at the selected scale.
#Use ±1.65 (90% confidence) instead of ±1.96:
#This may generate hotspots.

In [ ]:
active["giz_class"] = 0
active.loc[active["GiZ"] >= 1.65, "giz_class"] = 1
active.loc[active["GiZ"] <= -1.65, "giz_class"] = 2

In [ ]:
#standard practice check class balance
print(active["giz_class"].value_counts())
print(active["giz_class"].value_counts(normalize=True))

In [ ]:
#Still no significant hotspot
#Means ---->Spatial clustering exists (your Moran’s I was strong).
#However, clustering is dominated by oversupply concentration, not deficit concentration.
#Also....Dubai’s structural parking landscape at 250m resolution does not exhibit statistically significant demand-driven hotspots at 95% confidence.

In [ ]:
#seems my classification problem is binary, whether I like it or not.
#cannot run a meaningful 3-class model.


#

In [ ]:
# ==============================
# Create Binary Gi* Target
# ==============================

# 1 = Coldspot (oversupply cluster)
# 0 = Not significant

active["giz_binary"] = 0
active.loc[active["GiZ"] <= -1.96, "giz_binary"] = 1

print(active["giz_binary"].value_counts(normalize=True))

In [ ]:
# ==============================
# Define Predictors
# ==============================

X = active[["res_ratio", "road_density", "hubdist", "poi_density"]]
y = active["giz_binary"]


In [ ]:
# ==============================
# Train/Test Split
# ==============================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# ==============================
# Random Forest Classifier
# ==============================

from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

rf_clf.fit(X_train, y_train)


In [ ]:
# ==============================
# Evaluation
# ==============================

from sklearn.metrics import classification_report, accuracy_score

y_pred = rf_clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

##